***PROYECTO- PREDICCION DE CAPTURA DE MERLUZA NEGRA BAJO CONDICIONES DE CALENTAMIENTO GLOBAL***

In [18]:
# URLs a los archivos directamente desde GitHub (asegurate que estén en el repo)
url_captura = "https://raw.githubusercontent.com/CristianCouto/prediccion_merluza_negra/main/data/raw/captura-puerto-flota-2019.csv"
url_clima = "https://raw.githubusercontent.com/CristianCouto/prediccion_merluza_negra/main/data/raw/clima_ushuaia_rio_grande_2019.csv"
url_sst = "https://raw.githubusercontent.com/CristianCouto/prediccion_merluza_negra/main/data/processed/sst_anomaly_tdf_2019_mensual.csv"

# Leer directamente desde GitHub
df_captura = pd.read_csv(url_captura, encoding="latin1")
df_clima = pd.read_csv(url_clima)
df_sst = pd.read_csv(url_sst)


HTTPError: HTTP Error 404: Not Found

In [16]:
#  GENERACIÓN DE df_merged A PARTIR DE ARCHIVOS LOCALES

import pandas as pd

# 1. Cargar y procesar capturas de Merluza Negra en Ushuaia
df_captura = pd.read_csv("data/raw/captura-puerto-flota-2019.csv", encoding="latin1")

df_merluza = df_captura[
    (df_captura["puerto"].str.lower().str.contains("ushuaia")) &
    (df_captura["especie"].str.lower().str.contains("merluza negra"))
].copy()

df_merluza["mes"] = pd.to_datetime(df_merluza["fecha"]).dt.month
df_captura_mensual = df_merluza.groupby("mes")["captura"].sum().reset_index()

# 2. Cargar clima mensual (Ushuaia y Río Grande promediado)
df_clima = pd.read_csv("data/raw/clima_ushuaia_rio_grande_2019.csv")
df_clima_mensual = df_clima.groupby("mes").agg({
    "temperatura": "mean",
    "humedad": "mean",
    "precipitacion": "mean"
}).reset_index()

# 3. Cargar anomalía mensual de SST
df_sst = pd.read_csv("data/processed/sst_anomaly_tdf_2019_mensual.csv")

# 4. Unir todo por 'mes'
df_merged = df_captura_mensual.merge(df_clima_mensual, on="mes").merge(df_sst, on="mes")

# 5. Agregar columnas necesarias para filtros posteriores
df_merged["provincia"] = "tierra del fuego"
df_merged["especie_agrupada"] = "merluza negra"

# ✔️ Ya podés usar df_merged para filtros y modelos
df_merged.head()


FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/captura-puerto-flota-2019.csv'

In [14]:
# Aplicar filtros para quedarnos solo con capturas en Tierra del Fuego y especie Merluza Negra
df_proyecto = df_merged[
    (df_merged["provincia"].str.lower() == "tierra del fuego") &
    (df_merged["especie_agrupada"].str.lower() == "merluza negra")
].copy()

# Agrupar por mes para consolidar la información
df_modelo = df_proyecto.groupby("mes").agg({
    "captura": "sum",
    "precipitacion": "mean",
    "humedad": "mean",
    "temperatura": "mean"
}).reset_index()

df_proyecto

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/dataset_final.csv'

In [6]:
# Paso 1: Importar librerías de modelado y métricas
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt

# Paso 2: Definir variables predictoras (X) y variable objetivo (y)
X = df_modelo[["precipitacion", "humedad", "temperatura"]]
y = df_modelo["captura"]

# Paso 3: Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Paso 4: Crear y entrenar el modelo
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Paso 5: Realizar predicciones
y_pred = model.predict(X_test)

# Paso 6: Evaluar el modelo
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"MAE: {mae:.2f}")
print(f"R²: {r2:.2f}")

# Paso 7: Graficar resultados
plt.plot(y_test.values, label="Captura real", marker='o')
plt.plot(y_pred, label="Captura predicha", marker='o', linestyle='--')
plt.xlabel("Ejemplos")
plt.ylabel("Captura (kg)")
plt.title("Captura Real vs Predicha - Random Forest")
plt.legend()
plt.grid(True)
plt.show()


NameError: name 'df_modelo' is not defined

In [8]:
# Paso 1: Recuperar 'anom' y 'anom2' desde df_proyecto (original antes del groupby)
anom_por_mes = df_proyecto.groupby("mes").agg({
    "anom": "mean",
    "anom2": "mean"
}).reset_index()

# Paso 2: Agregar esas columnas al df_modelo
df_modelo = pd.merge(df_modelo, anom_por_mes, on="mes")

# Paso 3: Agregar también el 'mes' como variable predictora
X = df_modelo[["mes", "precipitacion", "humedad", "temperatura", "anom", "anom2"]]
y = df_modelo["captura"]

# Paso 4: Entrenar modelo con todos los datos
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)

# Paso 5: Hacer predicciones
y_pred = model.predict(X)

# Paso 6: Evaluar rendimiento
from sklearn.metrics import mean_absolute_error, r2_score
mae = mean_absolute_error(y, y_pred)
r2 = r2_score(y, y_pred)

print(f"MAE: {mae:.2f}")
print(f"R²: {r2:.2f}")

# Paso 7: Graficar resultados
import matplotlib.pyplot as plt

plt.plot(df_modelo["mes"], y, label="Captura real", marker='o')
plt.plot(df_modelo["mes"], y_pred, label="Captura predicha", marker='o', linestyle='--')
plt.xlabel("Mes")
plt.ylabel("Captura (kg)")
plt.title("Captura Real vs Predicha (modelo mejorado)")
plt.legend()
plt.grid(True)
plt.show()


NameError: name 'df_proyecto' is not defined

In [10]:
# Obtener importancia de cada variable
importancia = model.feature_importances_

# Mostrar en DataFrame ordenado
importancia_df = pd.DataFrame({
    "variable": X.columns,
    "importancia": importancia
}).sort_values(by="importancia", ascending=False)

print(importancia_df)


NameError: name 'model' is not defined

In [12]:
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.barplot(data=importancia_df, x="importancia", y="variable")
plt.title("Importancia de Variables - Random Forest")
plt.xlabel("Importancia")
plt.ylabel("Variable")
plt.grid(True)
plt.show()


NameError: name 'importancia_df' is not defined

<Figure size 800x500 with 0 Axes>

In [2]:
import pandas as pd
import matplotlib.pyplot as plt

# Cargar datos reales (asegurate de tenerlos en las rutas correctas)
sst = pd.read_csv("data/processed/sst_anomaly_tdf_2019_mensual.csv")
captura = pd.read_csv("data/raw/captura-puerto-flota-2019.csv", encoding="latin1")

# Filtrar Merluza Negra capturada en Ushuaia
df_merluza = captura[
    (captura["puerto"].str.lower().str.contains("ushuaia")) &
    (captura["especie"].str.lower().str.contains("merluza negra"))
].copy()

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/sst_anomaly_tdf_2019_mensual.csv'